# Week 4 — HPC Hardware Bottlenecks & the Roofline Model

> Parsing kernel traces, computing arithmetic intensity, and building roofline plots & Gantt charts from scratch.

---

## 1. Theory

### 1.1 Why we need a roofline

A modern GPU has two pools of resources that can each become the bottleneck of any kernel:

* **Compute**: e.g. an A100 SXM at FP16 with sparsity peaks at $\approx 312\,\mathrm{TFLOPS}$.
* **Memory bandwidth**: A100 HBM2e $\approx 2\,\mathrm{TB/s}$; L2 $\approx 5\,\mathrm{TB/s}$; SRAM $\approx 19\,\mathrm{TB/s}$.

A kernel that streams data faster than the SMs can crunch it sits in the **memory-bound** regime. A kernel that has plenty of arithmetic per byte sits in the **compute-bound** regime. The roofline (Williams, Waterman, Patterson, 2009) is the visual decision tool that classifies any kernel as one or the other.

### 1.2 Arithmetic intensity

$$I \;=\; \frac{\#\,\mathrm{FLOPs}}{\#\,\mathrm{bytes\;moved}} \quad \mathrm{[FLOP/byte]}$$

Concrete examples:
* SAXPY ($y = a x + y$): one mul + one add per 2 loads + 1 store = $\frac{2}{3 \cdot 4} = 0.17$ FLOP/B.
* Dense matmul $C = A B$ with $A \in \mathbb{R}^{M \times K}, B \in \mathbb{R}^{K \times N}$: $\approx 2MNK$ FLOPs, $4(MK + KN)$ bytes in FP32. Intensity is unbounded as $M, N, K$ grow.
* Attention softmax: low intensity if not fused; **flash-attention** reduces HBM traffic by tiling and recomputing.

### 1.3 The roofline equation

$$P_{\mathrm{attainable}}(I) \;=\; \min\!\bigl( P_{\mathrm{peak}}, \beta \cdot I \bigr)$$

The **ridge point**

$$I^* = \frac{P_{\mathrm{peak}}}{\beta}$$

separates regimes. For A100 + HBM, $I^* \approx 312\,\mathrm{TFLOPS} / 2\,\mathrm{TB/s} \approx 156$ FLOP/B. Anything below this ridge is memory-bound; anything above is compute-bound.

### 1.4 Multi-tier extension

Real workloads touch multiple tiers (SRAM, L2, HBM). The attainable performance is bounded by the **slowest** ceiling that the workload hits:

$$P(I) \;=\; \min\!\Bigl( P_{\mathrm{peak}}, \min_k \beta_k I_k \Bigr)$$

This gives multiple "roofs" on the chart; the kernel point sits below the lowest active roof.

### 1.5 Stalls & async pipelines

Kernels launch on CUDA streams. With multiple streams, copy and compute can overlap — but a stall on stream 0 cannot be hidden if stream 1 has nothing to do. The **Gantt chart** of kernel events per stream is the canonical visualization to spot serialization, missed-overlap opportunities, and synchronization barriers.

References
----------
* Williams, S. et al. (2009). *Roofline: an insightful visual performance model for multicore architectures.* CACM.
* Dao, T. et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness.* NeurIPS.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import csv
import tempfile
import numpy as np
import matplotlib.pyplot as plt

from src.profiler import (
    KernelTrace,
    RooflineModel,
    arithmetic_intensity,
    build_gantt_segments,
    default_a100_hierarchy,
    default_h100_hierarchy,
    parse_simple_csv,
)
from src.profiler.kernel_gantt import categorize_kernel

print("TensorLens — Week 4 notebook loaded")


## 2. Synthesize a realistic kernel trace

We build a representative trace of an inference forward pass: token embedding, four transformer blocks (each: 4× GEMMs for QKV + output projection, attention softmax, MLP up/down, GELU), and the final logits matmul. Each kernel has realistic FLOP counts and byte transfers for a 2k-token sequence at d_model=4096.

In [ ]:
# Realistic kernel trace for a single forward pass.
# Times in milliseconds, sizes in bytes/FLOPs.
T, D, FF = 2048, 4096, 16384  # seq_len, d_model, d_ff
HEADS, D_HEAD = 32, D // 32   # 128
BYTES_PER = 2                 # fp16

def gemm_kernel(name, M, N, K, start_ms, stream=0):
    flops = 2 * M * N * K
    # Bytes moved (no tiling reuse): A(MxK) + B(KxN) + C(MxN) — fp16
    bytes_dram = BYTES_PER * (M * K + K * N + M * N)
    bytes_l2 = bytes_dram * 0.5
    bytes_sram = bytes_dram * 0.2
    # Assume FP16 tensor cores at ~150 TFLOPS effective => derive duration
    duration_ms = flops / (150e12) * 1000
    return KernelTrace(
        name=name,
        start_seconds=start_ms / 1000.0,
        duration_seconds=duration_ms / 1000.0,
        flops=flops,
        bytes_dram=bytes_dram,
        bytes_l2=bytes_l2,
        bytes_sram=bytes_sram,
        stream_id=stream,
    )

def softmax_kernel(name, T, start_ms, stream=0):
    flops = 5 * T * T * HEADS  # roughly 5 ops per element of attention matrix
    bytes_dram = BYTES_PER * T * T * HEADS * 2
    duration_ms = bytes_dram / (1.5e12) * 1000  # bandwidth-bound at 1.5 TB/s effective
    return KernelTrace(
        name=name,
        start_seconds=start_ms / 1000.0,
        duration_seconds=duration_ms / 1000.0,
        flops=flops,
        bytes_dram=bytes_dram,
        bytes_l2=bytes_dram * 0.4,
        bytes_sram=bytes_dram * 0.1,
        stream_id=stream,
    )

def memcpy_kernel(name, n_bytes, start_ms, stream=1):
    duration_ms = n_bytes / (32e9) * 1000  # PCIe Gen4 ~ 32 GB/s
    return KernelTrace(
        name=name,
        start_seconds=start_ms / 1000.0,
        duration_seconds=duration_ms / 1000.0,
        flops=0.0,
        bytes_dram=float(n_bytes),
        bytes_l2=0.0,
        bytes_sram=0.0,
        stream_id=stream,
    )

trace = []
clock = 0.0
trace.append(memcpy_kernel("memcpy_h2d_input", n_bytes=T * 4, start_ms=clock, stream=1))
clock += 0.10

for layer in range(4):
    trace.append(gemm_kernel(f"gemm_qkv_l{layer}", T, 3 * D, D, clock))
    clock += trace[-1].duration_seconds * 1000
    trace.append(softmax_kernel(f"attention_softmax_l{layer}", T, clock))
    clock += trace[-1].duration_seconds * 1000
    trace.append(gemm_kernel(f"gemm_attn_out_l{layer}", T, D, D, clock))
    clock += trace[-1].duration_seconds * 1000
    trace.append(gemm_kernel(f"gemm_mlp_up_l{layer}", T, FF, D, clock))
    clock += trace[-1].duration_seconds * 1000
    trace.append(gemm_kernel(f"gemm_mlp_down_l{layer}", T, D, FF, clock))
    clock += trace[-1].duration_seconds * 1000

trace.append(gemm_kernel("gemm_lm_head", T, 50000, D, clock))
clock += trace[-1].duration_seconds * 1000
trace.append(memcpy_kernel("memcpy_d2h_logits", n_bytes=T * 50000 * 4, start_ms=clock, stream=1))

print(f"Synthesized {len(trace)} kernels over {clock:.2f} ms")
print("First few kernels:")
for k in trace[:5]:
    print(f"  {k.name:<28s}  duration={k.duration_seconds*1000:7.3f} ms  AI={arithmetic_intensity(k):.2f}")


## 3. CSV round-trip — exercising the parser

The parser supports any CSV exporting `kernel_name, start_ms, duration_ms, flops, bytes_dram, bytes_l2, bytes_sram, stream_id`. We write our synthetic trace to a tempfile and re-parse it to demonstrate.

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
    writer = csv.writer(f)
    writer.writerow(["kernel_name", "start_ms", "duration_ms", "flops",
                     "bytes_dram", "bytes_l2", "bytes_sram", "stream_id"])
    for k in trace:
        writer.writerow([
            k.name,
            k.start_seconds * 1000,
            k.duration_seconds * 1000,
            k.flops,
            k.bytes_dram,
            k.bytes_l2,
            k.bytes_sram,
            k.stream_id,
        ])
    csv_path = f.name

reparsed = parse_simple_csv(csv_path)
print(f"Round-trip: wrote {len(trace)} kernels, parsed {len(reparsed)} kernels")
assert len(reparsed) == len(trace)


## 4. Roofline plot — A100 80 GB SXM

We build the multi-tier roofline using the published A100 hierarchy and scatter every kernel against it.

In [ ]:
A100_PEAK_TFLOPS = 312e12  # FP16 tensor-core peak

model_a100 = RooflineModel(
    peak_flops_per_sec=A100_PEAK_TFLOPS,
    hierarchy=default_a100_hierarchy(),
    device_name="NVIDIA A100 80GB SXM (FP16)",
)

ridges = model_a100.ridge_points()
print("Ridge points (FLOP/B):")
for name, I_star in ridges.items():
    print(f"  {name:<30s}  I* = {I_star:7.2f}")

fig, ax = plt.subplots(figsize=(10, 6))
model_a100.render(trace, tier="dram", ai_range=(1e-1, 1e4), ax=ax)
plt.tight_layout()
plt.show()


## 5. Per-kernel classification: memory-bound vs compute-bound

For each kernel we compute its arithmetic intensity, its DRAM-roof ceiling, and check whether achieved performance equals the predicted ceiling. The ratio `achieved / ceiling` measures hardware utilization.

In [ ]:
rows = []
hbm_bw = next(t for t in default_a100_hierarchy() if t.name.startswith("HBM")).bandwidth_bytes_per_sec
for k in trace:
    AI = arithmetic_intensity(k, tier="dram")
    ceiling = min(A100_PEAK_TFLOPS, hbm_bw * AI) if np.isfinite(AI) else A100_PEAK_TFLOPS
    achieved = k.achieved_flops_per_sec
    util = achieved / ceiling if ceiling > 0 else 0.0
    regime = "compute-bound" if AI >= A100_PEAK_TFLOPS / hbm_bw else ("memory-bound" if AI > 0 else "n/a")
    rows.append((k.name, AI, achieved / 1e12, ceiling / 1e12, util, regime))

print(f"{'kernel':<28s}  {'AI':>8s}  {'achieved':>10s}  {'ceiling':>10s}  {'util':>6s}  {'regime':<15s}")
for name, AI, ach, ceil, util, regime in rows:
    ai_str = f"{AI:8.2f}" if np.isfinite(AI) else "    inf "
    print(f"{name:<28s}  {ai_str}  {ach:8.2f} TF  {ceil:8.2f} TF  {util:6.2f}  {regime}")


## 6. Gantt chart of the kernel pipeline

We group kernels by stream and render each stream as a row on the Gantt chart. Color encodes kernel category (`gemm`, `attention`, `memcpy`, etc.).

In [ ]:
gantt = build_gantt_segments(trace)
stream_ids = sorted(gantt.keys())

CATEGORY_COLORS = {
    "gemm": "#1f77b4",
    "attention": "#ff7f0e",
    "memcpy": "#2ca02c",
    "reduce": "#d62728",
    "elemwise": "#9467bd",
    "other": "#7f7f7f",
}

fig, ax = plt.subplots(figsize=(12, 1.4 * len(stream_ids) + 1))
for row, sid in enumerate(stream_ids):
    for ev in gantt[sid]:
        ax.broken_barh(
            [(ev.start_seconds * 1000, ev.duration_seconds * 1000)],
            (row - 0.4, 0.8),
            facecolors=CATEGORY_COLORS.get(ev.category, "#7f7f7f"),
            edgecolor="black",
            linewidth=0.5,
        )
ax.set_yticks(range(len(stream_ids)))
ax.set_yticklabels([f"stream {s}" for s in stream_ids])
ax.set_xlabel("time (ms)")
ax.set_title("Kernel pipeline Gantt chart — async streams")
# Legend
handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=cat) for cat, c in CATEGORY_COLORS.items()]
ax.legend(handles=handles, loc="upper right", fontsize=8, ncol=3)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()


## 7. A100 vs H100 — what changes?

We rerun the same trace against the H100 hierarchy (HBM3 @ 3.35 TB/s, peak FP16 TC ≈ 989 TFLOPS for the SXM). The relative slopes of memory-bound kernels shift; the ridge point moves to higher arithmetic intensity.

In [ ]:
model_h100 = RooflineModel(
    peak_flops_per_sec=989e12,
    hierarchy=default_h100_hierarchy(),
    device_name="NVIDIA H100 80GB SXM (FP16)",
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
model_a100.render(trace, tier="dram", ai_range=(1e-1, 1e4), ax=axes[0])
model_h100.render(trace, tier="dram", ai_range=(1e-1, 1e4), ax=axes[1])
plt.tight_layout()
plt.show()

a100_ridge = A100_PEAK_TFLOPS / next(t for t in default_a100_hierarchy() if t.name.startswith('HBM')).bandwidth_bytes_per_sec
h100_ridge = 989e12 / next(t for t in default_h100_hierarchy() if t.name.startswith('HBM')).bandwidth_bytes_per_sec
print(f"A100 ridge point: {a100_ridge:.1f} FLOP/B")
print(f"H100 ridge point: {h100_ridge:.1f} FLOP/B  (+{(h100_ridge/a100_ridge - 1)*100:.1f}% — H100 needs *more* arithmetic per byte to be compute-bound)")


## 8. Take-aways

1. **Arithmetic intensity is the universal kernel diagnostic.** Two metrics — FLOPs and bytes moved — capture nearly everything that matters for first-order kernel performance.
2. **Roofline tells you where to optimize.** Memory-bound kernels need fusion, tiling, or smaller dtype; compute-bound kernels need better algorithms or higher-precision tensor cores.
3. **The H100's higher peak FLOP/s shifts the ridge.** Memory-bound kernels are *more* memory-bound on H100 because compute grew faster than bandwidth — operator fusion (e.g. FlashAttention) matters more, not less, on newer hardware.
4. **Gantt charts reveal serialization.** Even with multiple streams, a critical path on one stream blocks the wall-clock completion of all work.

### References

* Williams, S., Waterman, A., Patterson, D. (2009). *Roofline: an insightful visual performance model for multicore architectures.* CACM.
* NVIDIA A100 Tensor Core GPU Architecture white paper (2020).
* NVIDIA H100 Tensor Core GPU Architecture white paper (2022).
* Dao, T. et al. (2022). *FlashAttention.* NeurIPS.
